# 06 · Request/reply with correlation IDs and timeouts

**Level:** intermediate · **Time:** 40–50 minutes · **Prerequisites:** notebooks 01–05.

Build a small square-number RPC service. Send a request through RabbitMQ, route its response back to
the client's reply queue, ignore an unrelated response, and observe a bounded timeout when no worker runs.

```text
client → request queue → worker
client ← reply queue   ← worker
```

`reply_to` tells the worker where to respond; `correlation_id` lets a client associate a reply with a
request. RPC still has network delays and partial failures. Prefer an asynchronous event/task when
the caller does not need an immediate answer. A timeout means the outcome is unknown, not that work was cancelled.


In [ ]:
from pathlib import Path
import sys
import json
import time
import pika

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course" / "rabbit.py").is_file()), None)
assert ROOT is not None, "Open Jupyter from the RabbitMQ course folder."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.rabbit import connect, unique_name, wait_for_message


## 1. Declare the request queue

Use a unique queue so another presenter cannot receive your requests. Our worker runs in a background
thread with its **own connection and channel**. Pika's connection/channel operations are not generally
thread-safe; never call a worker's channel from the notebook thread.


In [ ]:
from threading import Event, Thread
from queue import Queue
from uuid import uuid4

request_queue = unique_name("rpc.square")
with connect() as connection:
    connection.channel().queue_declare(queue=request_queue)
print("Requests:", request_queue)


## 2. Define a bounded worker

Validate the input and return a structured error for invalid requests. The response keeps the request's
correlation ID. The worker publishes with confirmation and acknowledges the request only after that succeeds.
There is still a crash window between publishing the response and acknowledging the request: duplicate
requests/replies remain possible. Squaring a number has no side effect, making retries harmless here.

The stop flag and 30-second maximum lifetime prevent an abandoned lesson from running indefinitely.
The worker's connection is created, serviced, and closed inside its own thread.


In [ ]:
def run_worker(stop, ready, errors):
    try:
        with connect() as connection:
            channel = connection.channel()
            channel.basic_qos(prefetch_count=1)
            channel.confirm_delivery()

            def respond(channel, method, properties, body):
                try:
                    value = json.loads(body)["number"]
                    if isinstance(value, bool) or not isinstance(value, int) or not -1000 <= value <= 1000:
                        raise ValueError("number must be an integer from -1000 to 1000")
                    result = {"ok": True, "result": value * value}
                except (ValueError, TypeError, KeyError) as exc:
                    result = {"ok": False, "error": str(exc)}
                if not properties.reply_to or not properties.correlation_id:
                    channel.basic_nack(method.delivery_tag, requeue=False)
                    return
                try:
                    channel.basic_publish(
                        exchange="", routing_key=properties.reply_to,
                        body=json.dumps(result).encode(), mandatory=True,
                        properties=pika.BasicProperties(correlation_id=properties.correlation_id,
                                                        content_type="application/json"),
                    )
                except pika.exceptions.UnroutableError:
                    # Client's exclusive reply queue disappeared. There is no useful place for this reply.
                    channel.basic_nack(method.delivery_tag, requeue=False)
                else:
                    channel.basic_ack(method.delivery_tag)

            tag = channel.basic_consume(queue=request_queue, on_message_callback=respond, auto_ack=False)
            ready.set()
            deadline = time.monotonic() + 30
            try:
                while not stop.is_set() and time.monotonic() < deadline:
                    connection.process_data_events(time_limit=0.1)
            finally:
                if channel.is_open:
                    channel.basic_cancel(tag)
    except Exception as exc:
        errors.put(exc)
        ready.set()


## 3. Define a client with a deadline

The broker-generated exclusive reply queue belongs to this client's connection and disappears when
it closes. A single connection can handle multiple requests using correlation IDs; this introductory
function handles one request per connection to make cleanup obvious.

For the demonstration, optionally insert an unrelated reply first. Discard replies that do not match
the current correlation ID. A real client with many outstanding calls needs a map of pending IDs instead.


In [ ]:
def call_square(number, timeout=5.0, inject_unrelated=False):
    correlation_id = str(uuid4())
    with connect() as connection:
        channel = connection.channel()
        reply_queue = channel.queue_declare(queue="", exclusive=True, auto_delete=True).method.queue
        channel.confirm_delivery()
        if inject_unrelated:
            channel.basic_publish(exchange="", routing_key=reply_queue, body=b'{"ok": true, "result": -1}',
                                  properties=pika.BasicProperties(correlation_id="old-request"), mandatory=True)
        channel.basic_publish(
            exchange="", routing_key=request_queue, body=json.dumps({"number": number}).encode(),
            properties=pika.BasicProperties(reply_to=reply_queue, correlation_id=correlation_id,
                                            content_type="application/json", expiration="10000"),
            mandatory=True,
        )
        deadline = time.monotonic() + timeout
        while time.monotonic() < deadline:
            method, properties, body = channel.basic_get(queue=reply_queue, auto_ack=False)
            if method is None:
                connection.sleep(0.05)
                continue
            channel.basic_ack(method.delivery_tag)
            if properties.correlation_id == correlation_id:
                return json.loads(body)
            print("Ignored unrelated correlation ID:", properties.correlation_id)
        raise TimeoutError(f"No response within {timeout}s; the request may still run.")


## 4. Run successful and invalid requests, then stop the worker

Predict: `12` yields `144`; a string yields a structured error. The fake reply with result `-1` must
not be mistaken for the answer. Start and stop the worker in one cell so it cannot be left running
while you pause to teach. Cleanup in `finally` also runs if an assertion fails.


In [ ]:
stop, ready, errors = Event(), Event(), Queue()
worker = Thread(target=run_worker, args=(stop, ready, errors), daemon=True)
worker.start()
try:
    assert ready.wait(timeout=10), "Worker did not become ready."
    if not errors.empty():
        raise errors.get()
    answer = call_square(12, inject_unrelated=True)
    invalid = call_square("twelve")
    assert answer == {"ok": True, "result": 144}, answer
    assert invalid["ok"] is False, invalid
    print("Valid request:", answer)
    print("Invalid request:", invalid)
finally:
    stop.set()
    worker.join(timeout=10)
assert not worker.is_alive(), "Worker has not stopped; restart the kernel before retrying."
if not errors.empty():
    raise errors.get()
print("Worker stopped.")


## 5. Failure drill: no worker available

The request queue still exists, so publishing succeeds. No worker consumes the request, and the caller
times out. Predict: does publisher confirmation prove that a worker ran? Does a timeout delete the request?
The answers are both no. Its 10-second message TTL limits queued waiting; it is not cancellation of work
already delivered to a worker. Cleanup below deletes this lesson's remaining request.


In [ ]:
started = time.monotonic()
try:
    call_square(9, timeout=0.6)
except TimeoutError as exc:
    print("Expected failure:", exc)
else:
    raise AssertionError("There should be no worker for this unique request queue.")
print("Elapsed seconds:", round(time.monotonic() - started, 2))


## 6. Exercise and suggested solution

Change the service to return the cube. Identify both code locations to change before editing:
the worker's calculation and the client's expected result (`12 ** 3 == 1728`). Keep correlation IDs,
validation, deadlines, and acknowledgement order intact. Rerun the worker definition and demonstration
cell together. The queued timed-out request may be delivered first; its deleted reply queue is handled
by the worker's `UnroutableError` path.

Explain how you would retry a payment request safely. A new correlation ID identifies a network attempt;
a stable **business idempotency key** identifies the payment across retries. Do not charge twice just
because a response was lost. Later Django and outbox lessons make this distinction practical.

## 7. Cleanup

The request queue is unique to this run; deleting it also removes the timed-out request. Reply queues
have already been removed when their owning connections closed.


In [ ]:
with connect() as connection:
    connection.channel().queue_delete(queue=request_queue)
print("RPC request queue removed.")


**Troubleshooting:** a timeout can mean no consumer, slow processing, lost response, or failed worker;
check the worker error queue and broker consumer count. If this notebook was interrupted mid-cell,
restart the kernel before rerunning. The demo worker deliberately has a finite lifetime. Do not share
the Pika connection between its thread and the notebook's thread.

Official references: [request/reply tutorial](https://www.rabbitmq.com/tutorials/tutorial-six-python),
[Pika threading FAQ](https://pika.readthedocs.io/en/stable/faq.html), and
[message TTL](https://www.rabbitmq.com/docs/ttl).
